In [68]:
import pandas as pd
import numpy as np

In [69]:
df_site = pd.read_excel(r'\\pwestmiadd001.file.core.windows.net\addazfile01\Shares\EPC\03. Protection Enhanced DTM\Site Assessment (SA)\Data\Round 38\Sharable\Ethiopia DTM - SA R38_Dataset_Sharable without GPS.xlsx', sheet_name='Sites')

In [70]:
df_south_et = pd.read_excel(r'\\pwestmiadd001.file.core.windows.net\addazfile01\Shares\EPC\03. Protection Enhanced DTM\Site Assessment (SA)\Data\Round 39\Sharable\Ethiopia DTM - R39_Dataset.xlsx')

In [71]:
df_site_south = pd.concat([df_site, df_south_et])

In [72]:
df_site = df_site_south

In [73]:
# df_site.to_excel('Merged data frame with south Ethiopia.xlsx')

In [74]:
df_site.columns.to_list()

['M-1658: Survey Date',
 'M-0489: Survey Round',
 'M-0445: Site ID',
 'M-0448: Site Name',
 '1.1.d.2: Site Alternate Name',
 '1.4.a.2: Is site open?',
 'M-0303: Region',
 'M-0304: Zone',
 'M-0305: Woreda',
 'M-0306: Kebele',
 'OCHA Region',
 'OCHA Region P-Code',
 'OCHA Zone',
 'OCHA Zone P-Code',
 'OCHA Woreda',
 'OCHA Woreda P-Code',
 '1.4.a.1: Site Open Date',
 'S-1835: Site Started',
 'xxxx: If reopened, when did it reopen?',
 'M-0337: Site Classification',
 'M-0342: Settlement/site type',
 'M-0342: If settlement/site type is other, Please specify',
 'S2302: If site type is collective center, what type of collective center?',
 'S2302: If the type of collective center is other, Please specify',
 'xxxx: If host community, what is the most common shelter occupancy agreement for IDPs in the host community?',
 '1.1.i.1: Is the site physically accessible by car?',
 'xxxx: If no, select all alternative options of physical accessibility',
 'xxxx: If no, select all alternative options of ph

In [75]:
df_site['M-0342: Settlement/site type'].value_counts()

M-0342: Settlement/site type
Host community           1882
Spontaneous camp/site     609
Collective center         174
Dispersed settlement       75
Planned camp/site          29
Name: count, dtype: int64

In [76]:
df_site['Recoded_comp'] = df_site['M-0342: Settlement/site type'].replace({'Spontaneous camp/site': 'In Camp', 
                                                                'Collective center': 'In Camp', 
                                                                'Planned camp/site': 'In Camp',
                                                                'Host community': 'Out of Camp',
                                                                'Dispersed settlement':'Out of Camp'})

In [77]:
df_site['Recoded_comp'].value_counts()

Recoded_comp
Out of Camp    1957
In Camp         812
Name: count, dtype: int64

In [78]:
df_Woreda_level = df_site.groupby([ 'OCHA Region','OCHA Region P-Code','OCHA Zone', 'OCHA Zone P-Code','OCHA Woreda', 'OCHA Woreda P-Code'])['M-0309: Total Number of IDP HHs'].sum().reset_index()

In [79]:
df_Woreda_level

,OCHA Region,OCHA Region P-Code,OCHA Zone,OCHA Zone P-Code,OCHA Woreda,OCHA Woreda P-Code,M-0309: Total Number of IDP HHs
0,Afar,ET02,Awsi /Zone 1,ET0201,Afambo,ET020104,380
1,Afar,ET02,Awsi /Zone 1,ET0201,Asayita,ET020103,602
2,Afar,ET02,Awsi /Zone 1,ET0201,Dubti,ET020101,964
3,Afar,ET02,Awsi /Zone 1,ET0201,Elidar,ET020102,115
4,Afar,ET02,Awsi /Zone 1,ET0201,Kori,ET020108,125
...,...,...,...,...,...,...,...
477,Tigray,ET01,Southern,ET0104,Maichew town,ET010409,1371
478,Tigray,ET01,Southern,ET0104,Mekhoni town,ET010412,779
479,Tigray,ET01,Southern,ET0104,Neqsege,ET010403,188
480,Tigray,ET01,Southern,ET0104,Raya Azebo,ET010406,2042


In [80]:
df_woreda_cross = pd.pivot_table(
    df_site,
    index=[
        'OCHA Region',
        'OCHA Region P-Code',
        'OCHA Zone',
        'OCHA Zone P-Code',
        'OCHA Woreda',
        'OCHA Woreda P-Code'
    ],
    columns='Recoded_comp',
    values='M-0309: Total Number of IDP HHs',
    aggfunc=['sum']
).reset_index()

In [81]:
# df_woreda_a['Recoded_comp'].value_counts()

In [82]:
df_woreda_cross =  pd.crosstab( index=[
    df_site['OCHA Region'],
    df_site['OCHA Region P-Code'],
    df_site['OCHA Zone'],
    df_site['OCHA Zone P-Code'],
    df_site['OCHA Woreda'],
    df_site['OCHA Woreda P-Code'],
        
    ],
columns= df_site['Recoded_comp'],
values=df_site['M-0309: Total Number of IDP HHs'],
                                 aggfunc='sum'
                                        ).reset_index()

In [83]:
df_woreda_cross

Recoded_comp,OCHA Region,OCHA Region P-Code,OCHA Zone,OCHA Zone P-Code,OCHA Woreda,OCHA Woreda P-Code,In Camp,Out of Camp
0,Afar,ET02,Awsi /Zone 1,ET0201,Afambo,ET020104,380.0,NaN
1,Afar,ET02,Awsi /Zone 1,ET0201,Asayita,ET020103,355.0,247.0
2,Afar,ET02,Awsi /Zone 1,ET0201,Dubti,ET020101,300.0,664.0
3,Afar,ET02,Awsi /Zone 1,ET0201,Elidar,ET020102,115.0,NaN
4,Afar,ET02,Awsi /Zone 1,ET0201,Kori,ET020108,40.0,85.0
...,...,...,...,...,...,...,...,...
477,Tigray,ET01,Southern,ET0104,Maichew town,ET010409,NaN,1371.0
478,Tigray,ET01,Southern,ET0104,Mekhoni town,ET010412,42.0,737.0
479,Tigray,ET01,Southern,ET0104,Neqsege,ET010403,NaN,188.0
480,Tigray,ET01,Southern,ET0104,Raya Azebo,ET010406,NaN,2042.0


In [84]:
df_woreda_cross['Total HH R38'] = df_woreda_cross[['In Camp', 'Out of Camp']].sum(axis=1)

In [85]:
#zonal_level = df_site.groupby(['M-0303: Region','OCHA Zone', 'OCHA Zone P-Code','Recoded_comp'], dropna=False)['M-0309: Total Number of IDP HHs'].sum().reset_index()

In [86]:
df_woreda_cross['Total HH R38'].sum()

559326.0

In [87]:
def calculate_sample_size(df, population_size_col):
    
    Z = 1.65  # Z-score for 90% confidence level
    P = 0.5   # Proportion (assumed maximum variability)
    d = 0.1  # Margin of error
    deff = 1.5  # Design Effect

    def sample_size_formula(N):
        
        numerator = (Z**2) * P * N * (1 - P) * deff
        denominator = (d**2) * (N - 1) + (Z**2) * P * (1 - P)

        
        if denominator == 0:
            return np.nan
        
        
        sample_size = np.ceil(numerator / denominator)
        return min(sample_size, N)  

   
    df['Sample_Size_at_woreda'] = df['Total HH R38'].apply(sample_size_formula)
    # df['Sample_Size_outofcamp'] = df['Out of Camp'].apply(sample_size_formula)
    
    return df



df = pd.DataFrame(df_woreda_cross)


df = calculate_sample_size(df, 'Population_Size')


In [88]:
df

Recoded_comp,OCHA Region,OCHA Region P-Code,OCHA Zone,OCHA Zone P-Code,OCHA Woreda,OCHA Woreda P-Code,In Camp,Out of Camp,Total HH R38,Sample_Size_at_woreda
0,Afar,ET02,Awsi /Zone 1,ET0201,Afambo,ET020104,380.0,NaN,380.0,87.0
1,Afar,ET02,Awsi /Zone 1,ET0201,Asayita,ET020103,355.0,247.0,602.0,92.0
2,Afar,ET02,Awsi /Zone 1,ET0201,Dubti,ET020101,300.0,664.0,964.0,96.0
3,Afar,ET02,Awsi /Zone 1,ET0201,Elidar,ET020102,115.0,NaN,115.0,65.0
4,Afar,ET02,Awsi /Zone 1,ET0201,Kori,ET020108,40.0,85.0,125.0,67.0
...,...,...,...,...,...,...,...,...,...,...
477,Tigray,ET01,Southern,ET0104,Maichew town,ET010409,NaN,1371.0,1371.0,98.0
478,Tigray,ET01,Southern,ET0104,Mekhoni town,ET010412,42.0,737.0,779.0,95.0
479,Tigray,ET01,Southern,ET0104,Neqsege,ET010403,NaN,188.0,188.0,76.0
480,Tigray,ET01,Southern,ET0104,Raya Azebo,ET010406,NaN,2042.0,2042.0,99.0


In [89]:
df['In Camp'].isna().sum()

250

In [90]:
df['Out of Camp'].isna().sum()

103

In [91]:
df['proportion_incamp'] = ((df['In Camp']/ df['Total HH R38'])).round(2)
df['proportion_out_of_camp'] = ((df['Out of Camp']/ df['Total HH R38'])).round(2)
df['Sum_proportion'] = df[['proportion_incamp','proportion_out_of_camp']].sum(axis=1)

In [92]:
df

Recoded_comp,OCHA Region,OCHA Region P-Code,OCHA Zone,OCHA Zone P-Code,OCHA Woreda,OCHA Woreda P-Code,In Camp,Out of Camp,Total HH R38,Sample_Size_at_woreda,proportion_incamp,proportion_out_of_camp,Sum_proportion
0,Afar,ET02,Awsi /Zone 1,ET0201,Afambo,ET020104,380.0,NaN,380.0,87.0,1.00,NaN,1.0
1,Afar,ET02,Awsi /Zone 1,ET0201,Asayita,ET020103,355.0,247.0,602.0,92.0,0.59,0.41,1.0
2,Afar,ET02,Awsi /Zone 1,ET0201,Dubti,ET020101,300.0,664.0,964.0,96.0,0.31,0.69,1.0
3,Afar,ET02,Awsi /Zone 1,ET0201,Elidar,ET020102,115.0,NaN,115.0,65.0,1.00,NaN,1.0
4,Afar,ET02,Awsi /Zone 1,ET0201,Kori,ET020108,40.0,85.0,125.0,67.0,0.32,0.68,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
477,Tigray,ET01,Southern,ET0104,Maichew town,ET010409,NaN,1371.0,1371.0,98.0,NaN,1.00,1.0
478,Tigray,ET01,Southern,ET0104,Mekhoni town,ET010412,42.0,737.0,779.0,95.0,0.05,0.95,1.0
479,Tigray,ET01,Southern,ET0104,Neqsege,ET010403,NaN,188.0,188.0,76.0,NaN,1.00,1.0
480,Tigray,ET01,Southern,ET0104,Raya Azebo,ET010406,NaN,2042.0,2042.0,99.0,NaN,1.00,1.0


In [93]:
df['In camp SS'] = (df['Sample_Size_at_woreda']* df['proportion_incamp']).round(0)
df['Out_of_camp_SS'] = (df['Sample_Size_at_woreda']* df['proportion_out_of_camp']).round(0)
df['Total_SS'] = df[['In camp SS', 'Out_of_camp_SS']].sum(axis=1)

In [94]:
df.columns.to_list()

['OCHA Region',
 'OCHA Region P-Code',
 'OCHA Zone',
 'OCHA Zone P-Code',
 'OCHA Woreda',
 'OCHA Woreda P-Code',
 'In Camp',
 'Out of Camp',
 'Total HH R38',
 'Sample_Size_at_woreda',
 'proportion_incamp',
 'proportion_out_of_camp',
 'Sum_proportion',
 'In camp SS',
 'Out_of_camp_SS',
 'Total_SS']

In [95]:
# df.to_excel('Sample_size_at_woreda_level.xlsx')

In [96]:
## Assumptionns = Samples selected per ss = 12 ## Assuming good cluster size 

In [97]:
df

Recoded_comp,OCHA Region,OCHA Region P-Code,OCHA Zone,OCHA Zone P-Code,OCHA Woreda,OCHA Woreda P-Code,In Camp,Out of Camp,Total HH R38,Sample_Size_at_woreda,proportion_incamp,proportion_out_of_camp,Sum_proportion,In camp SS,Out_of_camp_SS,Total_SS
0,Afar,ET02,Awsi /Zone 1,ET0201,Afambo,ET020104,380.0,NaN,380.0,87.0,1.00,NaN,1.0,87.0,NaN,87.0
1,Afar,ET02,Awsi /Zone 1,ET0201,Asayita,ET020103,355.0,247.0,602.0,92.0,0.59,0.41,1.0,54.0,38.0,92.0
2,Afar,ET02,Awsi /Zone 1,ET0201,Dubti,ET020101,300.0,664.0,964.0,96.0,0.31,0.69,1.0,30.0,66.0,96.0
3,Afar,ET02,Awsi /Zone 1,ET0201,Elidar,ET020102,115.0,NaN,115.0,65.0,1.00,NaN,1.0,65.0,NaN,65.0
4,Afar,ET02,Awsi /Zone 1,ET0201,Kori,ET020108,40.0,85.0,125.0,67.0,0.32,0.68,1.0,21.0,46.0,67.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477,Tigray,ET01,Southern,ET0104,Maichew town,ET010409,NaN,1371.0,1371.0,98.0,NaN,1.00,1.0,NaN,98.0,98.0
478,Tigray,ET01,Southern,ET0104,Mekhoni town,ET010412,42.0,737.0,779.0,95.0,0.05,0.95,1.0,5.0,90.0,95.0
479,Tigray,ET01,Southern,ET0104,Neqsege,ET010403,NaN,188.0,188.0,76.0,NaN,1.00,1.0,NaN,76.0,76.0
480,Tigray,ET01,Southern,ET0104,Raya Azebo,ET010406,NaN,2042.0,2042.0,99.0,NaN,1.00,1.0,NaN,99.0,99.0


#### Creating Cum sum column 

In [98]:
# df['cumulative_Total_HH_R38'] = df['Total HH R38'].cumsum()

In [99]:
# df.columns

In [100]:
df['RSS In Camp'] = ((df['In camp SS']*20)/100).round(0)
df['RSS Out of Camp'] = ((df['Out_of_camp_SS']*20)/100).round(0)
df['In Camp Effective Sample size'] = df[['RSS In Camp', 'In camp SS']].sum(axis=1)
df['Out of Camp Effective Sample size'] = df[['RSS Out of Camp', 'Out_of_camp_SS']].sum(axis=1)

In [101]:
df['In Camp Effective Sample size'].sum()

16157.0

In [102]:

# with pd.ExcelWriter('Sample_size_at_woreda_level_with_New.xlsx', engine='xlsxwriter') as writer:
#     df.to_excel(writer, sheet_name='Woreda Sample Size', index=False)

#     # Access the workbook and worksheet
#     workbook  = writer.book
#     worksheet = writer.sheets['Woreda Sample Size']

#     # Get the dimensions of the DataFrame
#     (max_row, max_col) = df.shape

#     # Create an Excel table
#     worksheet.add_table(0, 0, max_row, max_col - 1, {
#         'columns': [{'header': col} for col in df.columns],
#         'name': 'SampleSizeTable',
#         'style': 'Table Style Medium 9'
#     })

In [103]:
df

Recoded_comp,OCHA Region,OCHA Region P-Code,OCHA Zone,OCHA Zone P-Code,OCHA Woreda,OCHA Woreda P-Code,In Camp,Out of Camp,Total HH R38,Sample_Size_at_woreda,proportion_incamp,proportion_out_of_camp,Sum_proportion,In camp SS,Out_of_camp_SS,Total_SS,RSS In Camp,RSS Out of Camp,In Camp Effective Sample size,Out of Camp Effective Sample size
0,Afar,ET02,Awsi /Zone 1,ET0201,Afambo,ET020104,380.0,NaN,380.0,87.0,1.00,NaN,1.0,87.0,NaN,87.0,17.0,NaN,104.0,0.0
1,Afar,ET02,Awsi /Zone 1,ET0201,Asayita,ET020103,355.0,247.0,602.0,92.0,0.59,0.41,1.0,54.0,38.0,92.0,11.0,8.0,65.0,46.0
2,Afar,ET02,Awsi /Zone 1,ET0201,Dubti,ET020101,300.0,664.0,964.0,96.0,0.31,0.69,1.0,30.0,66.0,96.0,6.0,13.0,36.0,79.0
3,Afar,ET02,Awsi /Zone 1,ET0201,Elidar,ET020102,115.0,NaN,115.0,65.0,1.00,NaN,1.0,65.0,NaN,65.0,13.0,NaN,78.0,0.0
4,Afar,ET02,Awsi /Zone 1,ET0201,Kori,ET020108,40.0,85.0,125.0,67.0,0.32,0.68,1.0,21.0,46.0,67.0,4.0,9.0,25.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477,Tigray,ET01,Southern,ET0104,Maichew town,ET010409,NaN,1371.0,1371.0,98.0,NaN,1.00,1.0,NaN,98.0,98.0,NaN,20.0,0.0,118.0
478,Tigray,ET01,Southern,ET0104,Mekhoni town,ET010412,42.0,737.0,779.0,95.0,0.05,0.95,1.0,5.0,90.0,95.0,1.0,18.0,6.0,108.0
479,Tigray,ET01,Southern,ET0104,Neqsege,ET010403,NaN,188.0,188.0,76.0,NaN,1.00,1.0,NaN,76.0,76.0,NaN,15.0,0.0,91.0
480,Tigray,ET01,Southern,ET0104,Raya Azebo,ET010406,NaN,2042.0,2042.0,99.0,NaN,1.00,1.0,NaN,99.0,99.0,NaN,20.0,0.0,119.0


# Cummulative Sum from Site data 

In [104]:
df_site['cumulative_Total_HH_R38'] = df_site['M-0309: Total Number of IDP HHs'].cumsum()

In [105]:
df_site.columns.to_list()

['M-1658: Survey Date',
 'M-0489: Survey Round',
 'M-0445: Site ID',
 'M-0448: Site Name',
 '1.1.d.2: Site Alternate Name',
 '1.4.a.2: Is site open?',
 'M-0303: Region',
 'M-0304: Zone',
 'M-0305: Woreda',
 'M-0306: Kebele',
 'OCHA Region',
 'OCHA Region P-Code',
 'OCHA Zone',
 'OCHA Zone P-Code',
 'OCHA Woreda',
 'OCHA Woreda P-Code',
 '1.4.a.1: Site Open Date',
 'S-1835: Site Started',
 'xxxx: If reopened, when did it reopen?',
 'M-0337: Site Classification',
 'M-0342: Settlement/site type',
 'M-0342: If settlement/site type is other, Please specify',
 'S2302: If site type is collective center, what type of collective center?',
 'S2302: If the type of collective center is other, Please specify',
 'xxxx: If host community, what is the most common shelter occupancy agreement for IDPs in the host community?',
 '1.1.i.1: Is the site physically accessible by car?',
 'xxxx: If no, select all alternative options of physical accessibility',
 'xxxx: If no, select all alternative options of ph

In [106]:
df_site_re = df_site[['M-0489: Survey Round', 'M-0445: Site ID','M-0448: Site Name', '1.1.d.2: Site Alternate Name',
                      '1.4.a.2: Is site open?', 'M-0303: Region', 'M-0304: Zone', 'M-0305: Woreda', 'M-0306: Kebele',
                      'OCHA Region', 'OCHA Region P-Code', 'OCHA Zone', 'OCHA Zone P-Code', 'OCHA Woreda', 'OCHA Woreda P-Code',
                      'M-0309: Total Number of IDP HHs', 'M-0310: Total Number of IDP Individuals','Recoded_comp']]

In [107]:
df_site_re_in_camp = df_site_re[df_site_re['Recoded_comp'] == 'In Camp']

In [108]:
df_site_re_in_camp.columns

Index(['M-0489: Survey Round', 'M-0445: Site ID', 'M-0448: Site Name',
       '1.1.d.2: Site Alternate Name', '1.4.a.2: Is site open?',
       'M-0303: Region', 'M-0304: Zone', 'M-0305: Woreda', 'M-0306: Kebele',
       'OCHA Region', 'OCHA Region P-Code', 'OCHA Zone', 'OCHA Zone P-Code',
       'OCHA Woreda', 'OCHA Woreda P-Code', 'M-0309: Total Number of IDP HHs',
       'M-0310: Total Number of IDP Individuals', 'Recoded_comp'],
      dtype='object')

In [109]:
df_grouped = df_site_re_in_camp.groupby(['OCHA Woreda P-Code']).agg({
    'M-0309: Total Number of IDP HHs': 'sum',
    'M-0445: Site ID': 'count'
}).reset_index()

In [110]:
df_site_re_in_camp.columns

Index(['M-0489: Survey Round', 'M-0445: Site ID', 'M-0448: Site Name',
       '1.1.d.2: Site Alternate Name', '1.4.a.2: Is site open?',
       'M-0303: Region', 'M-0304: Zone', 'M-0305: Woreda', 'M-0306: Kebele',
       'OCHA Region', 'OCHA Region P-Code', 'OCHA Zone', 'OCHA Zone P-Code',
       'OCHA Woreda', 'OCHA Woreda P-Code', 'M-0309: Total Number of IDP HHs',
       'M-0310: Total Number of IDP Individuals', 'Recoded_comp'],
      dtype='object')

In [111]:
df_site_withCount_incamp = df_site_re_in_camp.merge(df_grouped, on='OCHA Woreda P-Code', how= 'left')

In [112]:
df_site_withCount_incamp = df_site_withCount_incamp.merge(df[['OCHA Woreda P-Code','In Camp Effective Sample size']], on= 'OCHA Woreda P-Code' , how='left' )

In [113]:
df_site_withCount_incamp = df_site_withCount_incamp.rename(columns={'M-0445: Site ID_y': 'Number of clusters', 'M-0309: Total Number of IDP HHs_x': 'M-0309: Total Number of IDP HHs'})

In [114]:
df_site_withCount_incamp.drop(columns=['M-0309: Total Number of IDP HHs_y'], inplace=True)

In [115]:
df_site_withCount_incamp.columns

Index(['M-0489: Survey Round', 'M-0445: Site ID_x', 'M-0448: Site Name',
       '1.1.d.2: Site Alternate Name', '1.4.a.2: Is site open?',
       'M-0303: Region', 'M-0304: Zone', 'M-0305: Woreda', 'M-0306: Kebele',
       'OCHA Region', 'OCHA Region P-Code', 'OCHA Zone', 'OCHA Zone P-Code',
       'OCHA Woreda', 'OCHA Woreda P-Code', 'M-0309: Total Number of IDP HHs',
       'M-0310: Total Number of IDP Individuals', 'Recoded_comp',
       'Number of clusters', 'In Camp Effective Sample size'],
      dtype='object')

In [116]:
# Step 1: Group by 'OCHA Woreda P-Code' and calculate total HHs per group
df_site_withCount_incamp['Total_HHs_Per_Woreda'] = df_site_withCount_incamp.groupby(
    'OCHA Woreda P-Code')['M-0309: Total Number of IDP HHs'].transform('sum')

# Step 2: Calculate percentage
df_site_withCount_incamp['HH_Percentage'] = ((
    df_site_withCount_incamp['M-0309: Total Number of IDP HHs'] /
    df_site_withCount_incamp['Total_HHs_Per_Woreda']
)*100).round(0)

# Optional: Drop the intermediate total column if not needed
df_site_withCount_incamp.drop(columns='Total_HHs_Per_Woreda', inplace=True)

# Display the result
print(df_site_withCount_incamp[['OCHA Woreda P-Code', 'M-0309: Total Number of IDP HHs', 'HH_Percentage']].head())

  OCHA Woreda P-Code  M-0309: Total Number of IDP HHs  HH_Percentage
0           ET010706                               95           20.0
1           ET010706                              106           22.0
2           ET010702                             2284           82.0
3           ET010704                               79          100.0
4           ET010706                              280           58.0


In [117]:
# Step 1: Round the percentage to nearest integer
df_site_withCount_incamp['HH_Percentage_Rounded'] = df_site_withCount_incamp['HH_Percentage'].round().astype(int)

# Step 2: Create a list of Site IDs repeated by the rounded percentage
df_site_withCount_incamp['Site_ID_Repeated'] = df_site_withCount_incamp.apply(
    lambda row: [row['M-0445: Site ID_x']] * row['HH_Percentage_Rounded'], axis=1
)

# Step 3: Explode the list into multiple rows
df_exploded = df_site_withCount_incamp.explode('Site_ID_Repeated')

# Optional: Rename the exploded column
df_exploded = df_exploded.rename(columns={'Site_ID_Repeated': 'Exploded_Site_ID'})

# View the result
print(df_exploded[['OCHA Woreda P-Code', 'Exploded_Site_ID']].head())

  OCHA Woreda P-Code Exploded_Site_ID
0           ET010706            TG625
0           ET010706            TG625
0           ET010706            TG625
0           ET010706            TG625
0           ET010706            TG625


In [118]:
# df_exploded.to_excel('Exploaded1.xlsx')

In [119]:

# Optional: reproducibility
# np.random.seed(42)

# Reset index to avoid duplicate index labels from explode()
df_exploded = df_exploded.reset_index(drop=True)

def assign_unique_random(g):
    n = len(g)
    if n <= 100:
        # Unique random numbers between 1–100
        nums = np.random.choice(np.arange(1, 101), size=n, replace=False)
    else:
        # If more than 100 rows, repeat the sequence 1–100 safely
        nums = np.tile(np.arange(1, 101), int(np.ceil(n / 100)))[:n]
        np.random.shuffle(nums)
    return pd.Series(nums, index=g.index)

# Apply by woreda
df_exploded['Random_Number'] = (
    df_exploded.groupby('OCHA Woreda P-Code', group_keys=False)
    .apply(assign_unique_random)
)

# ✅ Each row now has a single random number (no lists)
print(df_exploded[['OCHA Woreda P-Code', 'Exploded_Site_ID', 'Random_Number']].head())






  OCHA Woreda P-Code Exploded_Site_ID  Random_Number
0           ET010706            TG625             60
1           ET010706            TG625            100
2           ET010706            TG625             17
3           ET010706            TG625             35
4           ET010706            TG625             11


C:\Users\aadmassie\AppData\Local\Temp\ipykernel_15460\1629419208.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_unique_random)


In [120]:
# df_exploded.to_excel('Exploaded2.xlsx')

### Defining the Number of Cluster size 

In [121]:
df.columns.to_list()

['OCHA Region',
 'OCHA Region P-Code',
 'OCHA Zone',
 'OCHA Zone P-Code',
 'OCHA Woreda',
 'OCHA Woreda P-Code',
 'In Camp',
 'Out of Camp',
 'Total HH R38',
 'Sample_Size_at_woreda',
 'proportion_incamp',
 'proportion_out_of_camp',
 'Sum_proportion',
 'In camp SS',
 'Out_of_camp_SS',
 'Total_SS',
 'RSS In Camp',
 'RSS Out of Camp',
 'In Camp Effective Sample size',
 'Out of Camp Effective Sample size']

In [122]:
df_incamp = df_site_re[df_site_re['Recoded_comp'] == 'In Camp']

In [123]:
df.columns

Index(['OCHA Region', 'OCHA Region P-Code', 'OCHA Zone', 'OCHA Zone P-Code',
       'OCHA Woreda', 'OCHA Woreda P-Code', 'In Camp', 'Out of Camp',
       'Total HH R38', 'Sample_Size_at_woreda', 'proportion_incamp',
       'proportion_out_of_camp', 'Sum_proportion', 'In camp SS',
       'Out_of_camp_SS', 'Total_SS', 'RSS In Camp', 'RSS Out of Camp',
       'In Camp Effective Sample size', 'Out of Camp Effective Sample size'],
      dtype='object', name='Recoded_comp')

In [124]:
# df_site_withCount_incamp['Sampled Clusters'] = (df_site_withCount_incamp['M-0309: Total Number of IDP HHs']/12).round(0)

In [125]:
df

Recoded_comp,OCHA Region,OCHA Region P-Code,OCHA Zone,OCHA Zone P-Code,OCHA Woreda,OCHA Woreda P-Code,In Camp,Out of Camp,Total HH R38,Sample_Size_at_woreda,proportion_incamp,proportion_out_of_camp,Sum_proportion,In camp SS,Out_of_camp_SS,Total_SS,RSS In Camp,RSS Out of Camp,In Camp Effective Sample size,Out of Camp Effective Sample size
0,Afar,ET02,Awsi /Zone 1,ET0201,Afambo,ET020104,380.0,NaN,380.0,87.0,1.00,NaN,1.0,87.0,NaN,87.0,17.0,NaN,104.0,0.0
1,Afar,ET02,Awsi /Zone 1,ET0201,Asayita,ET020103,355.0,247.0,602.0,92.0,0.59,0.41,1.0,54.0,38.0,92.0,11.0,8.0,65.0,46.0
2,Afar,ET02,Awsi /Zone 1,ET0201,Dubti,ET020101,300.0,664.0,964.0,96.0,0.31,0.69,1.0,30.0,66.0,96.0,6.0,13.0,36.0,79.0
3,Afar,ET02,Awsi /Zone 1,ET0201,Elidar,ET020102,115.0,NaN,115.0,65.0,1.00,NaN,1.0,65.0,NaN,65.0,13.0,NaN,78.0,0.0
4,Afar,ET02,Awsi /Zone 1,ET0201,Kori,ET020108,40.0,85.0,125.0,67.0,0.32,0.68,1.0,21.0,46.0,67.0,4.0,9.0,25.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477,Tigray,ET01,Southern,ET0104,Maichew town,ET010409,NaN,1371.0,1371.0,98.0,NaN,1.00,1.0,NaN,98.0,98.0,NaN,20.0,0.0,118.0
478,Tigray,ET01,Southern,ET0104,Mekhoni town,ET010412,42.0,737.0,779.0,95.0,0.05,0.95,1.0,5.0,90.0,95.0,1.0,18.0,6.0,108.0
479,Tigray,ET01,Southern,ET0104,Neqsege,ET010403,NaN,188.0,188.0,76.0,NaN,1.00,1.0,NaN,76.0,76.0,NaN,15.0,0.0,91.0
480,Tigray,ET01,Southern,ET0104,Raya Azebo,ET010406,NaN,2042.0,2042.0,99.0,NaN,1.00,1.0,NaN,99.0,99.0,NaN,20.0,0.0,119.0


In [126]:

df_grouped = df.groupby(['OCHA Region', 'OCHA Region P-Code', 'OCHA Zone', 'OCHA Zone P-Code', 'OCHA Woreda','OCHA Woreda P-Code']).agg({
    'In Camp Effective Sample size': 'sum',
    # 'M-0309: Total Number of IDP HHs': 'sum',
    # 'M-0445: Site ID_x': 'count'
}).reset_index()


In [127]:
df_grouped['In Camp Effective Sample size'].sum()

16157.0

In [128]:
df_grouped['Required Sampled Clusters'] = (df_grouped['In Camp Effective Sample size']/15).round(0)

In [129]:
# df_grouped.to_excel('Woreda Level sample size.xlsx')

In [130]:
df_exploded.columns

Index(['M-0489: Survey Round', 'M-0445: Site ID_x', 'M-0448: Site Name',
       '1.1.d.2: Site Alternate Name', '1.4.a.2: Is site open?',
       'M-0303: Region', 'M-0304: Zone', 'M-0305: Woreda', 'M-0306: Kebele',
       'OCHA Region', 'OCHA Region P-Code', 'OCHA Zone', 'OCHA Zone P-Code',
       'OCHA Woreda', 'OCHA Woreda P-Code', 'M-0309: Total Number of IDP HHs',
       'M-0310: Total Number of IDP Individuals', 'Recoded_comp',
       'Number of clusters', 'In Camp Effective Sample size', 'HH_Percentage',
       'HH_Percentage_Rounded', 'Exploded_Site_ID', 'Random_Number'],
      dtype='object')

In [131]:
# import pandas as pd

# Ensure both dataframes have the same woreda code type (string)
df_exploded['OCHA Woreda P-Code'] = df_exploded['OCHA Woreda P-Code'].astype(str)
df_grouped['OCHA Woreda P-Code'] = df_grouped['OCHA Woreda P-Code'].astype(str)

# Merge the sample size information
df_merged = df_exploded.merge(
    df_grouped[['OCHA Woreda P-Code', 'Required Sampled Clusters']],
    on='OCHA Woreda P-Code',
    how='left'
)

# Sort within each woreda (descending random number)
df_merged = df_merged.sort_values(
    ['OCHA Woreda P-Code', 'Random_Number'], ascending=[True, False]
)

# Mark top N rows per woreda as "Selected"
df_merged['Selected'] = (
    df_merged.groupby('OCHA Woreda P-Code')
    .cumcount()
    .lt(df_merged['Required Sampled Clusters'])
    .map({True: 'Selected', False: 'Not Selected'})
)

# View result
print(df_merged[['OCHA Woreda P-Code', 'Exploded_Site_ID', 'Random_Number', 'Required Sampled Clusters', 'Selected']].head())


      OCHA Woreda P-Code Exploded_Site_ID  Random_Number  \
22702           ET010101           TG1048            100   
22667           ET010101           TG1048             99   
22699           ET010101           TG1048             98   
22674           ET010101           TG1048             97   
22680           ET010101           TG1048             96   

       Required Sampled Clusters      Selected  
22702                        4.0      Selected  
22667                        4.0      Selected  
22699                        4.0      Selected  
22674                        4.0      Selected  
22680                        4.0  Not Selected  


In [132]:
df_merged_incamp = pd.crosstab(
    index=[
        df_merged['OCHA Region'],
        df_merged['OCHA Region P-Code'],
        df_merged['OCHA Zone'],
        df_merged['OCHA Zone P-Code'],
        df_merged['OCHA Woreda'],
        df_merged['OCHA Woreda P-Code'],
        df_merged['Exploded_Site_ID']
    ],
    columns=df_merged['Selected']
).reset_index()

In [133]:
# df_merged_incamp.to_excel('Final Incamp samples_for_incamp_sample_.xlsx')
# df_grouped.to_excel ('Total Effective Sample Size for_In Camp.xlsx')

In [134]:
# import pandas as pd

with pd.ExcelWriter('Sampled_sites_Incomp_woreda_level.xlsx', engine='xlsxwriter') as writer:
    # Write the two DataFrames to separate sheets
    df_merged_incamp.to_excel(writer, sheet_name='Final Incamp Samples', index=False)
    df_grouped.to_excel(writer, sheet_name='Total Effective Sample Size', index=False)

    # Access the workbook and format each sheet as a table
    workbook = writer.book
    for sheet_name, df_data in {
        'Final Incamp Samples': df_merged_incamp,
        'Total Effective Sample Size': df_grouped
    }.items():
        worksheet = writer.sheets[sheet_name]
        (max_row, max_col) = df_data.shape
        column_settings = [{'header': column} for column in df_data.columns]
        worksheet.add_table(0, 0, max_row, max_col - 1, {
            'columns': column_settings,
            'name': sheet_name.replace(" ", "_"),
            'style': 'Table Style Medium 9'
        })